# 06 — FAST Machine Learning vs Random Walk

This optimized notebook replaces the previous slow Script 06.

It fits each candidate **once per inner-validation year**, predicts the whole validation year in one batch, selects the best configuration, then fits only the winner for the outer year.

Models: Ridge, Elastic Net, Random Forest, Extra Trees, HistGradientBoosting, and XGBoost if available.

Search dimensions: CHANGE vs LOG_CHANGE targets; TT_OWN, TT_CROSS, FULL_PIT features; EXPANDING vs RECENT_5Y training; Random-Walk shrinkage.

Core metrics: RMSE, MAE, Forecast Skill vs RW, Theil's U, DM p-value.

The 2024–2025 final holdout is never loaded.

In [ ]:
from pathlib import Path
import json, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as student_t
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

CURRENCIES=["USD","GBP","EUR","JPY"]
HORIZONS=[1,3,7]
VALIDATION_YEARS=[2019,2020,2021,2022,2023]
FEATURE_FAMILIES=["TT_OWN","TT_CROSS","FULL_PIT"]
TARGET_FORMS=["CHANGE","LOG_CHANGE"]
TRAIN_WINDOWS=["EXPANDING","RECENT_5Y"]
SHRINKAGE_GRID=[0.00,0.10,0.25,0.50,0.75,1.00]
RANDOM_STATE=42
print("FAST ML configuration loaded.")

In [ ]:
XGB_AVAILABLE=False
try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE=True
    print("XGBoost available; fast histogram mode will be used.")
except Exception:
    print("XGBoost unavailable; continuing without it.")

In [ ]:
def find_root():
    for root in [Path.cwd(),Path.cwd().parent,Path.cwd().parent.parent]:
        dev=root/"data"/"processed"/"02_development_model_dataset.csv"
        feat=root/"data"/"processed"/"02_feature_sets.json"
        rwf=root/"outputs"/"predictions"/"04_random_walk_predictions_2019_2023.csv"
        if dev.exists() and feat.exists() and rwf.exists(): return root.resolve()
    raise FileNotFoundError("Run Scripts 02 and 04 first.")

ROOT=find_root()
DEV_PATH=ROOT/"data"/"processed"/"02_development_model_dataset.csv"
FEATURE_PATH=ROOT/"data"/"processed"/"02_feature_sets.json"
RW_PATH=ROOT/"outputs"/"predictions"/"04_random_walk_predictions_2019_2023.csv"
TABLE_DIR=ROOT/"outputs"/"tables"; FIGURE_DIR=ROOT/"outputs"/"figures"; PRED_DIR=ROOT/"outputs"/"predictions"
for p in [TABLE_DIR,FIGURE_DIR,PRED_DIR]: p.mkdir(parents=True,exist_ok=True)

df=pd.read_csv(DEV_PATH,parse_dates=["Date"])
for h in HORIZONS:
    c=f"Target_Date_H{h}"
    if c in df.columns: df[c]=pd.to_datetime(df[c],errors="coerce")
rw=pd.read_csv(RW_PATH,parse_dates=["Forecast_Origin","Target_Date"])
with open(FEATURE_PATH,"r",encoding="utf-8") as f: FEATURE_SETS=json.load(f)
df=df.sort_values("Date").reset_index(drop=True)
rw=rw.sort_values(["Currency","Horizon","Forecast_Origin"]).reset_index(drop=True)
assert df["Date"].max() < pd.Timestamp("2024-01-01")
print("Development rows:",len(df)); print("RW rows:",len(rw)); print("PASS — holdout not loaded.")

In [ ]:
def rmse(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float); m=np.isfinite(y)&np.isfinite(p)
    return float(np.sqrt(np.mean((y[m]-p[m])**2))) if m.sum() else np.nan

def mae(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float); m=np.isfinite(y)&np.isfinite(p)
    return float(np.mean(np.abs(y[m]-p[m]))) if m.sum() else np.nan

def skill(m,r): return float(1-m/r) if np.isfinite(m) and np.isfinite(r) and r!=0 else np.nan

def theils_u(m,r): return float(m/r) if np.isfinite(m) and np.isfinite(r) and r!=0 else np.nan

def dm_test(y,model,rw_pred,horizon=1):
    y=np.asarray(y,float); model=np.asarray(model,float); rw_pred=np.asarray(rw_pred,float)
    mask=np.isfinite(y)&np.isfinite(model)&np.isfinite(rw_pred)
    y=y[mask]; model=model[mask]; rw_pred=rw_pred[mask]; n=len(y)
    if n<20: return np.nan,np.nan
    d=(y-rw_pred)**2-(y-model)**2; lag=max(0,int(horizon)-1); lrv=np.var(d,ddof=1)
    for k in range(1,lag+1):
        if len(d)<=k: break
        cov=np.cov(d[k:],d[:-k],ddof=1)[0,1]; lrv += 2*(1-k/(lag+1))*cov
    if not np.isfinite(lrv) or lrv<=0: return np.nan,np.nan
    stat=np.mean(d)/np.sqrt(lrv/n); p=2*student_t.sf(abs(stat),df=n-1)
    return float(stat),float(p)

In [ ]:
def build_model(key):
    if key=="RIDGE_0.1": return Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler()),("m",Ridge(alpha=0.1))])
    if key=="RIDGE_1": return Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler()),("m",Ridge(alpha=1.0))])
    if key=="RIDGE_10": return Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler()),("m",Ridge(alpha=10.0))])
    if key=="ELASTIC_NET": return Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler()),("m",ElasticNet(alpha=0.001,l1_ratio=0.2,max_iter=5000,random_state=RANDOM_STATE))])
    if key=="RANDOM_FOREST": return Pipeline([("imp",SimpleImputer(strategy="median")),("m",RandomForestRegressor(n_estimators=100,max_depth=6,min_samples_leaf=10,max_features=0.7,random_state=RANDOM_STATE,n_jobs=-1))])
    if key=="EXTRA_TREES": return Pipeline([("imp",SimpleImputer(strategy="median")),("m",ExtraTreesRegressor(n_estimators=120,max_depth=7,min_samples_leaf=8,max_features=0.8,random_state=RANDOM_STATE,n_jobs=-1))])
    if key=="HIST_GB": return Pipeline([("imp",SimpleImputer(strategy="median")),("m",HistGradientBoostingRegressor(learning_rate=0.04,max_iter=150,max_leaf_nodes=15,l2_regularization=2.0,random_state=RANDOM_STATE))])
    if key=="XGBOOST":
        if not XGB_AVAILABLE: return None
        return Pipeline([("imp",SimpleImputer(strategy="median")),("m",XGBRegressor(n_estimators=120,max_depth=3,learning_rate=0.04,subsample=0.8,colsample_bytree=0.8,reg_lambda=5.0,reg_alpha=0.1,objective="reg:squarederror",tree_method="hist",random_state=RANDOM_STATE,n_jobs=4,verbosity=0))])
    raise ValueError(key)

MODEL_KEYS=["RIDGE_0.1","RIDGE_1","RIDGE_10","ELASTIC_NET","RANDOM_FOREST","EXTRA_TREES","HIST_GB"]
if XGB_AVAILABLE: MODEL_KEYS.append("XGBOOST")
print("Models:",MODEL_KEYS)

In [ ]:
def target_name(cur,h,form): return f"{cur}_change_H{h}" if form=="CHANGE" else f"{cur}_log_change_H{h}"
def reconstruct(current,pred,form):
    current=np.asarray(current,float); pred=np.asarray(pred,float)
    return current+pred if form=="CHANGE" else current*np.exp(pred)

def make_batch(cur,h,form,family,window,val_year):
    feats=[c for c in FEATURE_SETS[cur][family] if c in df.columns]
    target=target_name(cur,h,form); tdate=f"Target_Date_H{h}"
    start=pd.Timestamp(f"{val_year}-01-01"); end=pd.Timestamp(f"{val_year+1}-01-01")
    train=df[(df["Date"]<start)&df[target].notna()&df[tdate].notna()&(df[tdate]<start)].copy()
    if window=="RECENT_5Y": train=train[train["Date"]>=start-pd.DateOffset(years=5)].copy()
    val=df[(df["Date"]>=start)&(df["Date"]<end)&df[target].notna()&df[tdate].notna()].copy()
    return train,val,feats,target

## Stage A — fast representation screening
Ridge screens target form, feature family, and training window using only the previous year.

In [ ]:
def screen_representation(cur,h,outer_year):
    inner=outer_year-1; rows=[]
    for form in TARGET_FORMS:
        for family in FEATURE_FAMILIES:
            for window in TRAIN_WINDOWS:
                train,val,feats,target=make_batch(cur,h,form,family,window,inner)
                if len(train)<200 or len(val)<30 or not feats: continue
                try:
                    model=build_model("RIDGE_1"); model.fit(train[feats],train[target])
                    pred=reconstruct(val[f"{cur}_mid"],model.predict(val[feats]),form)
                    rows.append({"Target_Form":form,"Feature_Family":family,"Train_Window":window,"Inner_RMSE":rmse(val[f"{cur}_target_H{h}"],pred)})
                except Exception: pass
    tab=pd.DataFrame(rows)
    if tab.empty: return None,tab
    b=tab.sort_values("Inner_RMSE").iloc[0]
    return {"Target_Form":b["Target_Form"],"Feature_Family":b["Feature_Family"],"Train_Window":b["Train_Window"]},tab

## Stage B — model screening
Each ML family is fitted once on the inner-training block and predicts the entire inner year.

In [ ]:
def screen_models(cur,h,outer_year,rep):
    inner=outer_year-1
    train,val,feats,target=make_batch(cur,h,rep["Target_Form"],rep["Feature_Family"],rep["Train_Window"],inner)
    if len(train)<200 or len(val)<30: return None,pd.DataFrame(),{}
    actual=val[f"{cur}_target_H{h}"].to_numpy(float); rw_pred=val[f"{cur}_mid"].to_numpy(float)
    rows=[]; preds={}
    for key in MODEL_KEYS:
        model=build_model(key)
        if model is None: continue
        try:
            t0=time.time(); model.fit(train[feats],train[target]); raw=reconstruct(rw_pred,model.predict(val[feats]),rep["Target_Form"])
            preds[key]=raw; rows.append({"Model":key,"Raw_Inner_RMSE":rmse(actual,raw),"Fit_Seconds":time.time()-t0})
        except Exception: pass
    return {"Actual":actual,"RW":rw_pred},pd.DataFrame(rows),preds

## Stage C — Random-Walk shrinkage and top-3 ensemble
This uses existing predictions, so it adds almost no fitting time.

In [ ]:
def select_inner(ctx,model_table,preds):
    if ctx is None or model_table.empty: return None,pd.DataFrame()
    actual=ctx["Actual"]; rw_pred=ctx["RW"]; rows=[]
    for key,raw in preds.items():
        for lam in SHRINKAGE_GRID:
            pred=rw_pred+lam*(raw-rw_pred)
            rows.append({"Final_Type":"SINGLE","Model_Key":key,"Lambda":lam,"Inner_RMSE":rmse(actual,pred)})
    top3=model_table.sort_values("Raw_Inner_RMSE").head(3)["Model"].tolist()
    if len(top3)>=2:
        raw=np.mean([preds[k] for k in top3],axis=0)
        for lam in SHRINKAGE_GRID:
            pred=rw_pred+lam*(raw-rw_pred)
            rows.append({"Final_Type":"ENSEMBLE","Model_Key":"|".join(top3),"Lambda":lam,"Inner_RMSE":rmse(actual,pred)})
    tab=pd.DataFrame(rows); b=tab.sort_values("Inner_RMSE").iloc[0]
    return {"Final_Type":b["Final_Type"],"Model_Key":b["Model_Key"],"Lambda":float(b["Lambda"]),"Inner_RMSE":float(b["Inner_RMSE"])},tab

In [ ]:
def predict_outer(cur,h,outer_year,rep,sel):
    train,outer,feats,target=make_batch(cur,h,rep["Target_Form"],rep["Feature_Family"],rep["Train_Window"],outer_year)
    rw_task=rw[(rw["Currency"]==cur)&(rw["Horizon"]==h)&(rw["Forecast_Origin"].dt.year==outer_year)][["Forecast_Origin","Target_Date","Actual","RW_Prediction"]].copy()
    outer=outer.merge(rw_task,left_on="Date",right_on="Forecast_Origin",how="inner")
    if outer.empty: return pd.DataFrame()
    raw_list=[]
    for key in sel["Model_Key"].split("|"):
        model=build_model(key); model.fit(train[feats],train[target]); raw_list.append(reconstruct(outer["RW_Prediction"],model.predict(outer[feats]),rep["Target_Form"]))
    raw=np.mean(raw_list,axis=0); rwv=outer["RW_Prediction"].to_numpy(float); final=rwv+sel["Lambda"]*(raw-rwv)
    return pd.DataFrame({"Currency":cur,"Horizon":h,"Validation_Year":outer_year,"Forecast_Origin":outer["Forecast_Origin"],"Target_Date":outer["Target_Date"],"Actual":outer["Actual"],"RW_Prediction":outer["RW_Prediction"],"Raw_ML_Prediction":raw,"ML_Prediction":final,"Target_Form":rep["Target_Form"],"Feature_Family":rep["Feature_Family"],"Train_Window":rep["Train_Window"],"Final_Type":sel["Final_Type"],"Selected_Model":sel["Model_Key"],"Shrinkage_Lambda":sel["Lambda"]})

## Run all 60 tasks
Progress and elapsed seconds are printed after every task.

In [ ]:
all_preds=[]; selections=[]; rep_tabs=[]; model_tabs=[]; cand_tabs=[]
task=0; total=60; start_all=time.time()
for cur in CURRENCIES:
    for h in HORIZONS:
        for year in VALIDATION_YEARS:
            task+=1; t0=time.time(); print(f"[{task:02d}/{total}] {cur} H{h} — {year}",flush=True)
            rep,rtab=screen_representation(cur,h,year)
            if rep is None: rep={"Target_Form":"CHANGE","Feature_Family":"TT_OWN","Train_Window":"EXPANDING"}
            if not rtab.empty:
                x=rtab.copy(); x["Currency"]=cur; x["Horizon"]=h; x["Outer_Year"]=year; rep_tabs.append(x)
            ctx,mtab,preds=screen_models(cur,h,year,rep); sel,ctab=select_inner(ctx,mtab,preds)
            if sel is None: sel={"Final_Type":"SINGLE","Model_Key":"RIDGE_1","Lambda":0.0,"Inner_RMSE":np.nan}
            if not mtab.empty:
                x=mtab.copy(); x["Currency"]=cur; x["Horizon"]=h; x["Outer_Year"]=year; model_tabs.append(x)
            if not ctab.empty:
                x=ctab.copy(); x["Currency"]=cur; x["Horizon"]=h; x["Outer_Year"]=year; cand_tabs.append(x)
            pred=predict_outer(cur,h,year,rep,sel); all_preds.append(pred)
            sec=time.time()-t0
            selections.append({"Currency":cur,"Horizon":h,"Outer_Year":year,"Inner_Validation_Year":year-1,**rep,"Final_Type":sel["Final_Type"],"Selected_Model":sel["Model_Key"],"Shrinkage_Lambda":sel["Lambda"],"Inner_RMSE":sel["Inner_RMSE"],"Task_Seconds":sec})
            print(f"   {sel['Model_Key']} | lambda={sel['Lambda']:.2f} | {sec:.1f}s",flush=True)
ml_predictions=pd.concat(all_preds,ignore_index=True); ml_selections=pd.DataFrame(selections)
print(f"\nFinished in {(time.time()-start_all)/60:.2f} minutes."); print("Prediction rows:",len(ml_predictions))

In [ ]:
yearly=[]
for (cur,h,year),g in ml_predictions.groupby(["Currency","Horizon","Validation_Year"]):
    mr=rmse(g["Actual"],g["ML_Prediction"]); ma=mae(g["Actual"],g["ML_Prediction"]); rr=rmse(g["Actual"],g["RW_Prediction"]); _,p=dm_test(g["Actual"],g["ML_Prediction"],g["RW_Prediction"],h)
    yearly.append({"Currency":cur,"Horizon":h,"Validation_Year":year,"Observations":len(g),"RMSE":mr,"MAE":ma,"Forecast_Skill_vs_RW":skill(mr,rr),"Theils_U":theils_u(mr,rr),"DM_pvalue_vs_RW":p})
yearly_results=pd.DataFrame(yearly).sort_values(["Currency","Horizon","Validation_Year"]); display(yearly_results)
yearly_results.to_csv(TABLE_DIR/"06_FAST_ml_yearly_results.csv",index=False)

In [ ]:
pooled=[]
for (cur,h),g in ml_predictions.groupby(["Currency","Horizon"]):
    mr=rmse(g["Actual"],g["ML_Prediction"]); ma=mae(g["Actual"],g["ML_Prediction"]); rr=rmse(g["Actual"],g["RW_Prediction"]); _,p=dm_test(g["Actual"],g["ML_Prediction"],g["RW_Prediction"],h)
    pooled.append({"Currency":cur,"Horizon":h,"Observations":len(g),"RMSE":mr,"MAE":ma,"Forecast_Skill_vs_RW":skill(mr,rr),"Theils_U":theils_u(mr,rr),"DM_pvalue_vs_RW":p})
pooled_results=pd.DataFrame(pooled).sort_values(["Currency","Horizon"]); display(pooled_results)
pooled_results.to_csv(TABLE_DIR/"06_FAST_ml_pooled_results.csv",index=False)
wins=pooled_results.assign(Beats_RW=pooled_results["Forecast_Skill_vs_RW"]>0,DM_Significant_5pct=pooled_results["DM_pvalue_vs_RW"]<0.05)
display(wins); print("Tasks beating RW:",int(wins["Beats_RW"].sum()),"/",len(wins))

In [ ]:
ml_predictions.to_csv(PRED_DIR/"06_FAST_ml_predictions_2019_2023.csv",index=False)
ml_selections.to_csv(TABLE_DIR/"06_FAST_ml_selections.csv",index=False)
if rep_tabs: pd.concat(rep_tabs,ignore_index=True).to_csv(TABLE_DIR/"06_FAST_representation_screen.csv",index=False)
if model_tabs: pd.concat(model_tabs,ignore_index=True).to_csv(TABLE_DIR/"06_FAST_model_screen.csv",index=False)
if cand_tabs: pd.concat(cand_tabs,ignore_index=True).to_csv(TABLE_DIR/"06_FAST_inner_candidates.csv",index=False)
display(ml_selections)

In [ ]:
rw_keys=set(zip(rw["Currency"],rw["Horizon"],rw["Forecast_Origin"])); ml_keys=set(zip(ml_predictions["Currency"],ml_predictions["Horizon"],ml_predictions["Forecast_Origin"]))
quality_gate=pd.DataFrame([
    {"Area":"Holdout isolation","Status":"PASS","Evidence":"Only pre-2024 development data loaded"},
    {"Area":"Frozen RW benchmark","Status":"PASS","Evidence":"Script 04 predictions reused"},
    {"Area":"Batch tuning","Status":"PASS","Evidence":"Candidates fitted once per inner year"},
    {"Area":"Target overlap control","Status":"PASS","Evidence":"Training target dates precede validation start"},
    {"Area":"Matched origins","Status":"PASS" if rw_keys==ml_keys else "REVIEW","Evidence":f"RW={len(rw_keys)}, ML={len(ml_keys)}"},
    {"Area":"Selection","Status":"PASS","Evidence":"Previous-year inner validation only"},
    {"Area":"Shrinkage","Status":"PASS","Evidence":"Selected using inner predictions only"}
])
display(quality_gate); quality_gate.to_csv(TABLE_DIR/"06_FAST_quality_gate.csv",index=False)

In [ ]:
print("="*78); print("SCRIPT 06 FAST ML COMPLETE"); print("="*78)
display(pooled_results)
print("Forecast Skill > 0 and Theil U < 1 = lower RMSE than RW.")
print("DM p < 0.05 = stronger statistical evidence.")
print("2024–2025 remains untouched.")